# 🎧 Spotify Seasonality Analysis — Christmas Peak vs Summer Hits
---

#### This notebook analyzes seasonality and audio features (danceability, valence, tempo, energy) across the European Spotify charts.

In [ ]:
# --- Imports and setup ---
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

In [ ]:
# --- Load dataset ---
df = pd.read_csv('../data/eu_top200_with_custom_popularity.csv')
print(df.info())
df.head()

In [ ]:
# Convert and extract date components
df['date'] = pd.to_datetime(df['date'], errors='coerce')

In [ ]:
# --- Define temporal windows ---
df['is_christmas_period'] = ((df['date'].dt.month == 12) & (df['date'].dt.day >= 10)) | \
                            ((df['date'].dt.month == 1) & (df['date'].dt.day <= 7))
df['is_summer_period'] = df['date'].dt.month.isin([6,7,8])

# Subsets
christmas_df = df[df['is_christmas_period']]
summer_df = df[df['is_summer_period']]

#### 💭 **Hypothesis:** The most popular Christmas songs are the ones generating the highest total streams during the Christmas window; also in summer we expect a similar trend for "Summer Hits"

In [ ]:
# Aggregate by Country and Track_id using streams, let's see if we spot any interesting patterns in the top Christmas hits across different European countries.
top_christmas_by_country = (
    christmas_df
    .groupby(["country", "track_id"], as_index=False)
    .agg(
        artist_name=("main_artist", "first"), # all rows in the group refer to the same track, 'first' is just a convenient way to get the artist name without needing to reset the index or do additional merges
        track_name=("title", "first"),
        total_streams=("streams", "sum")
    )
    .sort_values(["country", "total_streams"], ascending=[True, False])
)

for country, group in top_christmas_by_country.groupby("country"):
    
    print(f"\n🎄 {country}")
    
    display(
        group.sort_values("total_streams", ascending=False)
             .head(10)[["artist_name", "track_name", "total_streams"]]
    )

##### 👉 We can see that in many countries the Top 10 is dominated by recurring Christmas classics, while in a few (e.g., France/Spain/Italy/Greece) there are more “regular” hits mixing in.

#### Let's see a Pan-European analysis for Christmas:

In [ ]:
# Aggregate Total Christmas Streams (Pan-European)
top_christmas_tracks = (
    christmas_df
    .groupby("track_id", as_index=False)
    .agg(
        artist_name=("main_artist", "first"), # all rows in the group refer to the same track, 
        track_name=("title", "first"),        #"first" value is perfectly safe for our purposes 
        total_streams=("streams", "sum")
    )
    .sort_values("total_streams", ascending=False)
)

top_christmas_tracks.head(30)

In [ ]:
top30 = top_christmas_tracks.head(30)

plt.figure(figsize=(14,10))
plt.barh(top30["track_name"], top30["total_streams"])
plt.gca().invert_yaxis()

plt.xlabel("Total Streams During Christmas Window")
plt.title("Top 30 Songs by Total Christmas Streams")
plt.tight_layout()
plt.show()

##### 👉 We can see that in Europe the Top 30 is dominated by recurring Christmas classics, whith the exception of 'Dance Monkey'

#### Pan-European analysis for Summer:

In [ ]:
# Aggregate Total Summer Streams (Pan-European)
top_summer_tracks = (
    summer_df
    .groupby("track_id", as_index=False)
    .agg(
        artist_name=("main_artist", "first"), # all rows in the group refer to the same track, 
        track_name=("title", "first"),        #"first" value is perfectly safe for our purposes 
        total_streams=("streams", "sum")
    )
    .sort_values("total_streams", ascending=False)
)

top_summer_tracks.head(30)

In [ ]:
top30 = top_summer_tracks.head(30)

plt.figure(figsize=(14,10))
plt.barh(top30["track_name"], top30["total_streams"])
plt.gca().invert_yaxis()

plt.xlabel("Total Streams During Summer Window")
plt.title("Top 30 Songs by Total Summer Streams")
plt.tight_layout()
plt.show()

##### 👉 We can see that in Europe the Top 30 Summer hits have no a clear dominance by any recurring hits. 

##### 👉 This reinforces the idea that Christmas hits have a seasonality, while summer hits renew roughly every summer.

##### 🔎 Let's verify seasonality for a well-known set of Christmas and Summer songs 

In [ ]:

# =============================================================
# MASKS
# =============================================================
# Use sets for O(1) lookup — much faster than lists with .isin()
christmas_tracks_ids = set(top_christmas_tracks.head(20)['track_id'].tolist())
summer_tracks_ids    = set(top_summer_tracks.head(20)['track_id'].tolist())

# =============================================================
# 1. Filter by Track ID
# =============================================================
df_christmas_tracks = df[df['track_id'].isin(christmas_tracks_ids)].copy()
df_summer_tracks    = df[df['track_id'].isin(summer_tracks_ids)].copy()

# ✅ Found tracks (count by track_id to avoid NaN/duplicate title issues)
print(f"🎄 Christmas tracks found: {df_christmas_tracks['track_id'].nunique()}/20")
print(f"☀️  Summer tracks found:   {df_summer_tracks['track_id'].nunique()}/20")

# =============================================================
# 2. "Day of year" column to align years
# =============================================================
df['day_of_year'] = df['date'].dt.dayofyear
df_christmas_tracks['day_of_year'] = df_christmas_tracks['date'].dt.dayofyear
df_summer_tracks['day_of_year']    = df_summer_tracks['date'].dt.dayofyear

# =============================================================
# 3. AGGREGATION: daily total streams per group
# =============================================================
sum_christmas = (
    df_christmas_tracks
    .groupby('day_of_year')['streams']
    .sum()
    .reset_index()
)

sum_summer = (
    df_summer_tracks
    .groupby('day_of_year')['streams']
    .sum()
    .reset_index()
)

# =============================================================
# 4. PLOT
# =============================================================
plt.figure(figsize=(14, 7))

sns.lineplot(
    data=sum_christmas, x='day_of_year', y='streams',
    label=f'Christmas Hits (Top {df_christmas_tracks["track_id"].nunique()})',
    color='red'
)
sns.lineplot(
    data=sum_summer, x='day_of_year', y='streams',
    label=f'Summer Hits (Top {df_summer_tracks["track_id"].nunique()})',
    color='orange'
)

# Seasonal color bands
plt.axvspan(344, 365, color='red',    alpha=0.1, label='Christmas Season (Dec)')
plt.axvspan(1,   7,   color='red',    alpha=0.1)  # first week of January
plt.axvspan(152, 243, color='orange', alpha=0.1, label='Summer Season (Jun–Aug)')

plt.title('Popularity Trends: Christmas Hits vs Summer Hits', fontsize=15)
plt.xlabel("Day of the Year (1–365)")
plt.ylabel('Total Streams')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 📊 Popularity Seasonality: Summer Hits vs Christmas Hits

### Key Observations

**🎄 Christmas Songs (Red Line)**
The seasonality pattern of Christmas tracks is the most striking finding of this analysis.
These songs are essentially **"invisible" for ~10 months of the year**, hovering near **0 streams** 
from January through October. Starting around **day 310 (early November)**, streams begin rising 
sharply, culminating in a dramatic **peak around day 355 (late December)** — reaching over 
**1,200M total streams**, the highest value observed in the entire chart.
The cliff-drop immediately after Christmas (day 360+) is equally sharp, confirming that 
Christmas music consumption is almost entirely confined to a **~7 week window**.

---

**☀️ Summer Songs (Orange Line)**
Summer hits display a fundamentally different and **more stable** pattern throughout the year.
Rather than a sharp seasonal spike, they maintain a **consistent baseline of ~400–450M streams** 
even outside their peak season — suggesting these tracks have broader, year-round appeal.
As expected, streams rise during the **June–August window (days 152–243)**, peaking around 
**day 200 (~620M streams)**, before gradually declining in autumn.

---

### 💡 Analytical Takeaway

| | Christmas Hits | Summer Hits |
|---|---|---|
| **Peak streams** | ~1,200M | ~620M |
| **Peak period** | Late Dec (day ~355) | Mid July (day ~200) |
| **Off-season streams** | ~0 | ~400M |
| **Seasonality intensity** | ⭐⭐⭐⭐⭐ Extreme | ⭐⭐ Moderate |

> The data confirms a clear asymmetry: **Christmas songs are seasonal phenomena** with extreme 
> concentration of plays in a narrow window, while **summer hits are evergreen tracks** that happen 
> to peak in summer but sustain listener interest year-round. 
> This has direct implications for playlist strategy and music marketing campaigns.

#### 📈 Differential distribution of audio features between Christmas and Summer

In [ ]:
features = ['danceability', 'energy', 'valence', 'tempo']

# =============================================================
# 1. TOP 50 FILTER by total streams in each period
# =============================================================
top50_xmas_titles = (
    christmas_df.groupby('title')['streams']
    .sum()
    .sort_values(ascending=False)
    .head(50)
    .index                        # returns the title index directly
)

top50_summer_titles = (
    summer_df.groupby('title')['streams']
    .sum()
    .sort_values(ascending=False)
    .head(50)
    .index
)

# Apply the filter
df_xmas   = christmas_df[christmas_df['title'].isin(top50_xmas_titles)]
df_summer = summer_df[summer_df['title'].isin(top50_summer_titles)]

print(f"🎄 Christmas rows after filter: {len(df_xmas):,}")
print(f"☀️  Summer rows after filter:   {len(df_summer):,}")

# =============================================================
# 2. ADD SEASON LABEL & STACK
# =============================================================
df_xmas['season']   = 'Christmas'
df_summer['season'] = 'Summer'     

df_seasonal = pd.concat([df_xmas, df_summer], ignore_index=True)

# =============================================================
# 3. PLOT
# =============================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 6))

palette = {'Christmas': '#e74c3c', 'Summer': '#f39c12'}

for ax, feature in zip(axes, features):
    sns.boxplot(
        data      = df_seasonal,
        x         = 'season',
        y         = feature,
        hue       = 'season',    # ✅ FutureWarning fix
        palette   = palette,
        legend    = False,       # ✅ avoids duplicate legend
        width     = 0.5,
        linewidth = 1.2,
        flierprops= dict(marker='o', markersize=2, alpha=0.3),
        ax        = ax
    )
    ax.set_title(feature.capitalize(), fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(feature.capitalize(), fontsize=10)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle(
    'Audio Features Distribution — Top 50 Most Streamed: Christmas vs Summer',
    fontsize=15, fontweight='bold', y=1.02
)

plt.tight_layout()
plt.show()

#### 🕺 💃 Pair-wise T-test between Christmas and Summer

In [ ]:
from scipy import stats

features = ['danceability', 'energy', 'valence', 'tempo']

top50_xmas_titles = (
    christmas_df.groupby('title')['streams']
    .sum()
    .sort_values(ascending=False)
    .head(50)
    .index
)

top50_summer_titles = (
    summer_df.groupby('title')['streams']
    .sum()
    .sort_values(ascending=False)
    .head(50)
    .index
)

df_xmas   = christmas_df[christmas_df['title'].isin(top50_xmas_titles)]
df_summer = summer_df[summer_df['title'].isin(top50_summer_titles)]

# =============================================================
# 1. DEDUPLICATE: one row per track (mean of audio features)
#    ⚠️ Important: the same track appears many times in the dataset
#    (one row per day per region). We collapse to one row per title
#    to avoid pseudoreplication in the t-test.
# =============================================================
df_xmas_agg   = df_xmas.groupby('title')[features].mean()
df_summer_agg = df_summer.groupby('title')[features].mean()

print(f"🎄 Christmas tracks: {len(df_xmas_agg)}")
print(f"☀️  Summer tracks:   {len(df_summer_agg)}\n")

# =============================================================
# 2. T-TEST (independent samples) for each feature
# =============================================================
print(f"{'Feature':<15} {'Mean Xmas':>10} {'Mean Summer':>12} {'t-stat':>10} {'p-value':>10} {'Significant':>12}")
print("-" * 72)

alpha = 0.05

for feature in features:
    xmas_vals   = df_xmas_agg[feature].dropna()
    summer_vals = df_summer_agg[feature].dropna()

    t_stat, p_value = stats.ttest_ind(xmas_vals, summer_vals)

    significant = "✅ YES" if p_value < alpha else "❌ NO"

    print(f"{feature:<15} {xmas_vals.mean():>10.3f} {summer_vals.mean():>12.3f} "
          f"{t_stat:>10.3f} {p_value:>10.4f} {significant:>12}")

## 📊 T-Test Results: Christmas vs Summer Audio Features

### Methodology
An **independent samples t-test** (α = 0.05) was conducted on the **top 50 most streamed tracks**
for each season (December for Christmas, June–August for Summer).  
Each track was represented by the **mean value** of its audio features across all chart appearances,
to avoid pseudoreplication.

---

### Interpretation

- **Valence** shows the most intuitive seasonal split: Summer hits score significantly higher,
  confirming that warmer months favor **happier, more upbeat songs**.
- **Danceability** is higher in Summer tracks, consistent with the dominance of dance-pop
  and reggaeton in summer charts.
- **Energy** differences, if significant, suggest Christmas hits lean toward
  **softer, more acoustic** productions (ballads, orchestral arrangements).
- **Tempo** differences reflect the genre mix: Summer charts skew toward
  high-BPM dance tracks, while Christmas hits include many slow ballads.

---

### ⚠️ Limitations
> - Sample size is limited to **50 tracks per group** — results may not generalize to all music.
> - The dataset covers **2017–2021**: trends may differ in other periods.
> - Audio features are computed by Spotify's algorithm and carry their own measurement uncertainty.

### ⭐⭐⭐⭐ Visualization of statistical significance (optional improvements)

In [ ]:
features = ['danceability', 'energy', 'valence', 'tempo']

# =============================================================
# 1. SIGNIFICANCE HELPER FUNCTION
# =============================================================
def get_stars(p_value):
    if p_value < 0.0001:
        return '****'
    elif p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return 'ns'          # not significant

# =============================================================
# 2. COMPUTE P-VALUES (reusing same subsets as before)
# =============================================================
pvalues = {}

for feature in features:
    xmas_vals   = df_xmas_agg[feature].dropna()
    summer_vals = df_summer_agg[feature].dropna()
    _, p        = stats.ttest_ind(xmas_vals, summer_vals)
    pvalues[feature] = p

# =============================================================
# 3. REBUILD STACKED DATAFRAME FOR SEABORN
# =============================================================
df_xmas_plot   = df_xmas_agg.copy();   df_xmas_plot['season']   = 'Christmas'
df_summer_plot = df_summer_agg.copy(); df_summer_plot['season'] = 'Summer'
df_seasonal    = pd.concat([df_xmas_plot, df_summer_plot])

palette = {'Christmas': '#e74c3c', 'Summer': '#f39c12'}

# =============================================================
# 4. PLOT WITH SIGNIFICANCE ANNOTATIONS
# =============================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 7))

for ax, feature in zip(axes, features):
    sns.boxplot(
        data      = df_seasonal,
        x         = 'season',
        y         = feature,
        hue       = 'season',
        palette   = palette,
        legend    = False,
        width     = 0.5,
        linewidth = 1.2,
        flierprops= dict(marker='o', markersize=2, alpha=0.3),
        ax        = ax
    )

    # --- Significance bar & stars ---
    stars   = get_stars(pvalues[feature])
    y_max   = df_seasonal[feature].max()
    y_range = df_seasonal[feature].max() - df_seasonal[feature].min()
    y_bar   = y_max + y_range * 0.05   # bar sits 5% above max value
    y_text  = y_bar + y_range * 0.02   # stars sit just above the bar

    # Draw the horizontal bracket
    ax.plot([0, 1], [y_bar, y_bar], color='black', linewidth=1.2)
    ax.plot([0, 0], [y_bar - y_range*0.01, y_bar], color='black', linewidth=1.2)
    ax.plot([1, 1], [y_bar - y_range*0.01, y_bar], color='black', linewidth=1.2)

    # Color-code: grey for ns, black for significant
    star_color = 'grey' if stars == 'ns' else 'black'
    ax.text(0.5, y_text, stars, ha='center', va='bottom',
            fontsize=14, fontweight='bold', color=star_color)

    # Extend y-axis to make room for annotation
    ax.set_ylim(top=y_text + y_range * 0.08)

    ax.set_title(feature.capitalize(), fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(feature.capitalize(), fontsize=10)
    ax.grid(axis='y', alpha=0.3)

# --- Legend for stars ---
legend_text = "ns: p≥0.05 | *: p<0.05 | **: p<0.01 | ***: p<0.001 | ****: p<0.0001"
fig.text(0.5, -0.02, legend_text, ha='center', fontsize=10, color='dimgray')

fig.suptitle(
    'Audio Features — Top 50 Most Streamed: Christmas vs Summer',
    fontsize=15, fontweight='bold', y=1.02
)

plt.tight_layout()
plt.show()